In [1]:
import psycopg2
import psycopg2.extras
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import folium
from shapely.geometry import Point, Polygon, MultiPolygon
from shapely.ops import unary_union
import os
import random
import warnings
from datetime import datetime, timedelta
warnings.filterwarnings('ignore')
 
print("Sve biblioteke uspešno uvezene!")

Sve biblioteke uspešno uvezene!


In [2]:
# Povezivanje na bazu podataka

DB_CONFIG = {
    'dbname': 'sumski_pozari_db',
    'user': 'postgres',
    'password': 'milica123',  
    'host': 'localhost',
    'port': '5432'
}

conn = psycopg2.connect(**DB_CONFIG)
cur = conn.cursor()
print(" Povezivanje na bazu uspešno!")

# Provera PostGIS-a
cur.execute("SELECT postgis_version();")
version = cur.fetchone()
print(f" PostGIS verzija: {version[0]}")

 Povezivanje na bazu uspešno!
 PostGIS verzija: 3.6 USE_GEOS=1 USE_PROJ=1 USE_STATS=1


In [3]:
# Kreiranje tabela u bazi podataka

tables = ['opozarena_podrucja', 'pozari', 'meteo_merenja',
          'meteo_stanice', 'vatrogasne_jedinice', 'sumske_parcele', 'opstine']
 
for table in tables:
    cur.execute(f"DROP TABLE IF EXISTS {table} CASCADE;")
conn.commit()
print("Stare tabele obrisane")
 
cur.execute("""
CREATE TABLE opstine (
    id               SERIAL PRIMARY KEY,
    naziv            VARCHAR(100) NOT NULL,
    region           VARCHAR(100),
    povrsina_km2     NUMERIC(10,2),
    broj_stanovnika  INTEGER,
    geom             GEOMETRY(MultiPolygon, 4326)
);
""")
 
cur.execute("""
CREATE TABLE sumske_parcele (
    id                 SERIAL PRIMARY KEY,
    naziv              VARCHAR(100) NOT NULL,
    tip_sume           VARCHAR(50),
    povrsina_ha        NUMERIC(10,2),
    gustina_sume       VARCHAR(20),
    nadmorska_visina   INTEGER,
    opstina_id         INTEGER REFERENCES opstine(id),
    geom               GEOMETRY(Geometry, 4326)   
);
""")
 
cur.execute("""
CREATE TABLE vatrogasne_jedinice (
    id             SERIAL PRIMARY KEY,
    naziv          VARCHAR(100) NOT NULL,
    adresa         TEXT,
    broj_vozila    INTEGER,
    broj_osoblja   INTEGER,
    radno_vreme    VARCHAR(20),
    opstina_id     INTEGER REFERENCES opstine(id),
    geom           GEOMETRY(Point, 4326)
);
""")
 
cur.execute("""
CREATE TABLE meteo_stanice (
    id           SERIAL PRIMARY KEY,
    naziv        VARCHAR(100) NOT NULL,
    visina_m     INTEGER,
    tip_stanice  VARCHAR(20),
    opstina_id   INTEGER REFERENCES opstine(id),
    geom         GEOMETRY(Point, 4326)
);
""")
 
cur.execute("""
CREATE TABLE meteo_merenja (
    id                SERIAL PRIMARY KEY,
    stanica_id        INTEGER REFERENCES meteo_stanice(id),
    datum             DATE NOT NULL,
    temperatura       NUMERIC(5,2),
    vlaznost_vazduha  NUMERIC(5,2),
    brzina_vetra      NUMERIC(5,2),
    padavine_mm       NUMERIC(5,2)
);
""")
 
cur.execute("""
CREATE TABLE pozari (
    id                              SERIAL PRIMARY KEY,
    datum_pocetka                   DATE NOT NULL,
    datum_gasenja                   DATE,
    parcela_id                      INTEGER REFERENCES sumske_parcele(id),
    uzrok                           TEXT,
    intenzitet                      VARCHAR(20),
    povrsina_zahvacena_ha           NUMERIC(10,2),
    broj_angazovanih_vatrogasaca    INTEGER
);
""")
 
# Brisanje požara automatski briše i njegova opožarena područja
cur.execute("""
CREATE TABLE opozarena_podrucja (
    id                SERIAL PRIMARY KEY,
    pozar_id          INTEGER REFERENCES pozari(id) ON DELETE CASCADE,
    datum_detekcije   DATE,
    metoda_detekcije  VARCHAR(50),
    povrsina_ha       NUMERIC(10,2),
    indeks_dnbr       NUMERIC(5,4),
    geom              GEOMETRY(Geometry, 4326)   -- prima Polygon i MultiPolygon
);
""")
 
conn.commit()
print("Sve tabele uspešno kreirane!")

Stare tabele obrisane
Sve tabele uspešno kreirane!


In [4]:
# Unos podataka u tabelu opštine

opstine = [
    # naziv, region, povrsina_km2, stanovnika, MULTIPOLYGON (lon_min, lat_min, lon_max, lat_max)
    ("Pirot", "Stara planina", 1232.00, 56700,
     "MULTIPOLYGON(((22.45 43.10, 22.78 43.10, 22.78 43.30, 22.45 43.30, 22.45 43.10)))"),
 
    ("Babušnica", "Stara planina", 524.00, 9800,
     "MULTIPOLYGON(((22.30 43.00, 22.55 43.00, 22.55 43.20, 22.30 43.20, 22.30 43.00)))"),
 
    ("Dimitrovgrad", "Stara planina", 480.00, 8300,
     "MULTIPOLYGON(((22.62 42.92, 22.92 42.92, 22.92 43.12, 22.62 43.12, 22.62 42.92)))"),
 
    ("Knjaževac", "Stara planina", 1202.00, 28400,
     "MULTIPOLYGON(((22.10 43.42, 22.50 43.42, 22.50 43.70, 22.10 43.70, 22.10 43.42)))"),
 
    ("Zaječar", "Timočka regija", 1059.00, 50700,
     "MULTIPOLYGON(((22.10 43.78, 22.50 43.78, 22.50 44.05, 22.10 44.05, 22.10 43.78)))"),
]
 
for naziv, region, povrsina, stanovnika, geom in opstine:
    cur.execute("""
        INSERT INTO opstine (naziv, region, povrsina_km2, broj_stanovnika, geom)
        VALUES (%s, %s, %s, %s, ST_GeomFromText(%s, 4326))
    """, (naziv, region, povrsina, stanovnika, geom))
 
conn.commit()
print("Uneto 5 opština")


Uneto 5 opština


In [5]:
# Unos podataka u tabelu šumske parcele

parcele = [
    # naziv, tip, povrsina_ha, gustina, nadmorska, opstina_id, POLYGON
    ("Parcela Vlasina",        "bukova šuma",   340.50, "gusta",   1100, 1,
     "POLYGON((22.50 43.12, 22.62 43.12, 22.62 43.22, 22.50 43.22, 22.50 43.12))"),
   
    ("Parcela Rsovci",         "mesovita šuma", 210.20, "srednja",  850, 1,
     "POLYGON((22.62 43.12, 22.75 43.12, 22.75 43.22, 22.62 43.22, 22.62 43.12))"),
 
    ("Parcela Petrlaš", "borova šuma", 150.00, "retka", 950, 2,
     "POLYGON((22.32 43.05, 22.50 43.05, 22.50 43.18, 22.32 43.18, 22.32 43.05))"),
 
    ("Parcela Strelac", "bukova šuma", 275.80, "gusta", 1200, 3,
      "POLYGON((22.68 42.96, 22.85 42.96, 22.85 43.08, 22.68 43.08, 22.68 42.96))"),
 
    ("Parcela Stara planina sever", "mesovita šuma", 410.00, "gusta", 1350, 4,
     "POLYGON((22.15 43.45, 22.38 43.45, 22.38 43.62, 22.15 43.62, 22.15 43.45))"),
 
    ("Parcela Crni vrh",       "borova šuma",   198.40, "srednja", 1050, 5,
     "POLYGON((22.15 43.82, 22.38 43.82, 22.38 43.98, 22.15 43.98, 22.15 43.82))"),
]
 
for naziv, tip, povrsina, gustina, nadmorska, opstina_id, geom in parcele:
    cur.execute("""
        INSERT INTO sumske_parcele
        (naziv, tip_sume, povrsina_ha, gustina_sume, nadmorska_visina, opstina_id, geom)
        VALUES (%s, %s, %s, %s, %s, %s, ST_GeomFromText(%s, 4326))
    """, (naziv, tip, povrsina, gustina, nadmorska, opstina_id, geom))
 
conn.commit()
print("Uneto 6 šumskih parcela")

Uneto 6 šumskih parcela


In [6]:
# Unos podataka u tabelu vatrogasne jedinice

vatrogasne = [
    ("VJ Pirot",        "Nikole Pašića 12, Pirot",            6, 24, "24h", 1, "POINT(22.60 43.18)"),
    ("VJ Babušnica",    "Centar 5, Babušnica",                 2,  8, "24h", 2, "POINT(22.35 43.14)"),
    ("VJ Dimitrovgrad", "Glavna 3, Dimitrovgrad",              3, 10, "24h", 3, "POINT(22.85 43.02)"),
    ("VJ Knjaževac",    "Stevana Sinđelića 1, Knjaževac",      4, 14, "24h", 4, "POINT(22.28 43.56)"),
    ("VJ Zaječar",      "Trg oslobođenja 2, Zaječar",          7, 22, "24h", 5, "POINT(22.28 43.90)"),
]
 
for naziv, adresa, vozila, osoblje, radno_vreme, opstina_id, geom in vatrogasne:
    cur.execute("""
        INSERT INTO vatrogasne_jedinice
        (naziv, adresa, broj_vozila, broj_osoblja, radno_vreme, opstina_id, geom)
        VALUES (%s, %s, %s, %s, %s, %s, ST_GeomFromText(%s, 4326))
    """, (naziv, adresa, vozila, osoblje, radno_vreme, opstina_id, geom))
 
conn.commit()
print("Uneto 5 vatrogasnih jedinica")

Uneto 5 vatrogasnih jedinica


In [7]:
# Unos podataka u tabelu meteorološke stanice

stanice = [
    ("Meteo Pirot",              380,  "automatska", 1, "POINT(22.60 43.18)"),
    ("Meteo Babušnica",          450,  "automatska", 2, "POINT(22.35 43.14)"),
    ("Meteo Dimitrovgrad",       420,  "automatska", 3, "POINT(22.85 43.02)"),
    ("Meteo Knjaževac",          510,  "automatska", 4, "POINT(22.28 43.55)"),
    ("Meteo Stara planina vrh", 1500,  "ručna",      4, "POINT(22.22 43.46)"),
    ("Meteo Zaječar",            310,  "automatska", 5, "POINT(22.28 43.90)"),
]
 
for naziv, visina, tip, opstina_id, geom in stanice:
    cur.execute("""
        INSERT INTO meteo_stanice (naziv, visina_m, tip_stanice, opstina_id, geom)
        VALUES (%s, %s, %s, %s, ST_GeomFromText(%s, 4326))
    """, (naziv, visina, tip, opstina_id, geom))
 
conn.commit()
print("Uneto 6 meteo stanica")

Uneto 6 meteo stanica


In [8]:
# Unos podataka u tabelu meteorološka merenja

# Generišemo više meteo podataka za ceo period
random.seed(42) 
 
start_date = datetime(2024, 6, 1)
end_date   = datetime(2024, 10, 31)
 
dates = []
d = start_date
while d <= end_date:
    dates.append(d)
    d += timedelta(days=1)
 
# Broj stanica je sada 6 (dodata Zaječar)
cur.execute("SELECT COUNT(*) FROM meteo_stanice")
broj_stanica = cur.fetchone()[0]
 
meteo_podaci = [
    (stanica_id, datum,
     round(random.uniform(15, 40), 1),
     round(random.uniform(15, 85), 1),
     round(random.uniform(0, 45),  1),
     round(random.uniform(0, 20),  1))
    for datum in dates
    for stanica_id in range(1, broj_stanica + 1)
]
 
# execute_values — bulk insert, mnogo brži od petlje
psycopg2.extras.execute_values(cur, """
    INSERT INTO meteo_merenja
    (stanica_id, datum, temperatura, vlaznost_vazduha, brzina_vetra, padavine_mm)
    VALUES %s
""", meteo_podaci)
 
conn.commit()
print(f" Uneto {len(meteo_podaci)} meteo merenja ")

 Uneto 918 meteo merenja 


In [9]:
# Unos podataka u tabelu šumski požari

pozari = [
    # datum_poc,    datum_gas,    parcela_id, uzrok,                            intenzitet, povrsina, vatrogasci
    ("2024-07-10", "2024-07-12", 1, "ljudski faktor (loza)",                    "visok",   45.30, 18),
    ("2024-07-22", "2024-07-22", 2, "udar groma",                               "nizak",    3.10,  6),
    ("2024-08-02", "2024-08-05", 3, "ljudski faktor (spaljivanje korova)",       "srednji", 22.70, 12),
    ("2024-08-03", "2024-08-06", 4, "nepoznat",                                 "visok",   60.00, 25),  
    ("2024-08-10", "2024-08-12", 5, "nepoznat",                                 "visok",   55.00, 22),  
    ("2024-08-15", "2024-08-15", 6, "udar groma",                               "nizak",    5.50,  8),
]
 
for datum_poc, datum_gas, parcela_id, uzrok, intenzitet, povrsina, vatrogasci in pozari:
    cur.execute("""
        INSERT INTO pozari
        (datum_pocetka, datum_gasenja, parcela_id, uzrok, intenzitet,
         povrsina_zahvacena_ha, broj_angazovanih_vatrogasaca)
        VALUES (%s, %s, %s, %s, %s, %s, %s)
    """, (datum_poc, datum_gas, parcela_id, uzrok, intenzitet, povrsina, vatrogasci))
 
conn.commit()
print("Uneto 6 požara (sve parcele pokrivene)")

Uneto 6 požara (sve parcele pokrivene)


In [10]:
# Unos podataka u tabelu opožarena područja

opozarena = [
    # pozar_id, datum,         metoda,          povrsina, dnbr,  geom (unutar parcele!)
    (1, "2024-07-13", "NBR_threshold", 44.10, 0.412,
     "POLYGON((22.51 43.13, 22.60 43.13, 22.60 43.21, 22.51 43.21, 22.51 43.13))"),
 
    (2, "2024-07-23", "NBR_threshold",  3.00, 0.355,
     "POLYGON((22.63 43.13, 22.68 43.13, 22.68 43.18, 22.63 43.18, 22.63 43.13))"),
 
    (3, "2024-08-06", "NBR_threshold", 21.90, 0.480,
     "POLYGON((22.33 43.06, 22.48 43.06, 22.48 43.16, 22.33 43.16, 22.33 43.06))"),
 
    (4, "2024-08-07", "NBR_threshold", 58.70, 0.520,
     "POLYGON((22.69 42.97, 22.84 42.97, 22.84 43.07, 22.69 43.07, 22.69 42.97))"),
 
    (5, "2024-08-11", "NBR_threshold", 52.00, 0.495,
     "POLYGON((22.16 43.46, 22.36 43.46, 22.36 43.60, 22.16 43.60, 22.16 43.46))"),
 
    (6, "2024-08-16", "NBR_threshold",  5.20, 0.330,
     "POLYGON((22.16 43.83, 22.35 43.83, 22.35 43.96, 22.16 43.96, 22.16 43.83))"),
]
 
for pozar_id, datum, metoda, povrsina, indeks, geom in opozarena:
    cur.execute("""
        INSERT INTO opozarena_podrucja
        (pozar_id, datum_detekcije, metoda_detekcije, povrsina_ha, indeks_dnbr, geom)
        VALUES (%s, %s, %s, %s, %s, ST_GeomFromText(%s, 4326))
    """, (pozar_id, datum, metoda, povrsina, indeks, geom))
 
conn.commit()
print("Uneto 6 opožarenih područja (unutar granica parcela)")

Uneto 6 opožarenih područja (unutar granica parcela)


In [11]:
# Provera unosa podataka u sve tabele

tabele = ['opstine', 'sumske_parcele', 'vatrogasne_jedinice', 
          'meteo_stanice', 'meteo_merenja', 'pozari', 'opozarena_podrucja']

for tabela in tabele:
    cur.execute(f"SELECT COUNT(*) FROM {tabela}")
    broj = cur.fetchone()[0]
    print(f" {tabela}: {broj} redova")

print("\n Svi podaci uspešno uneti!")

 opstine: 5 redova
 sumske_parcele: 6 redova
 vatrogasne_jedinice: 5 redova
 meteo_stanice: 6 redova
 meteo_merenja: 918 redova
 pozari: 6 redova
 opozarena_podrucja: 6 redova

 Svi podaci uspešno uneti!


In [12]:
#zatvaranje konekcije

cur.close()
conn.close()
print("Konekcija zatvorena")

Konekcija zatvorena


In [13]:
#ponovo povezivanje na bazu

conn = psycopg2.connect(**DB_CONFIG)
cur = conn.cursor()
print("Ponovo povezani na bazu!")

Ponovo povezani na bazu!


In [14]:
#crud operacije za tabele

def citaj_sve_pozare():
    """Čita sve požare sa detaljima"""
    query = """
        SELECT p.id, p.datum_pocetka, p.datum_gasenja, 
               s.naziv as parcela, p.uzrok, p.intenzitet,
               p.povrsina_zahvacena_ha, p.broj_angazovanih_vatrogasaca
        FROM pozari p
        LEFT JOIN sumske_parcele s ON p.parcela_id = s.id
        ORDER BY p.datum_pocetka DESC
    """
    return pd.read_sql(query, conn)

# Prikaz svih požara
df_pozari = citaj_sve_pozare()
print("Svi požari:")
display(df_pozari)

Svi požari:


,id,datum_pocetka,datum_gasenja,parcela,uzrok,intenzitet,povrsina_zahvacena_ha,broj_angazovanih_vatrogasaca
0,6,2024-08-15,2024-08-15,Parcela Crni vrh,udar groma,nizak,5.5,8
1,5,2024-08-10,2024-08-12,Parcela Stara planina sever,nepoznat,visok,55.0,22
2,4,2024-08-03,2024-08-06,Parcela Strelac,nepoznat,visok,60.0,25
3,3,2024-08-02,2024-08-05,Parcela Petrlaš,ljudski faktor (spaljivanje korova),srednji,22.7,12
4,2,2024-07-22,2024-07-22,Parcela Rsovci,udar groma,nizak,3.1,6
5,1,2024-07-10,2024-07-12,Parcela Vlasina,ljudski faktor (loza),visok,45.3,18


In [15]:
# Crud operacija create

def dodaj_pozar(conn, cur, datum_pocetka, datum_gasenja, parcela_id,
                uzrok, intenzitet, povrsina, vatrogasci):
    # Validacija
    if datum_gasenja and datum_gasenja < datum_pocetka:
        print("Datum gašenja ne može biti pre datuma početka!")
        return None
    if povrsina <= 0:
        print("Površina mora biti pozitivna!")
        return None
 
    try:
        cur.execute("""
            INSERT INTO pozari
            (datum_pocetka, datum_gasenja, parcela_id, uzrok, intenzitet,
             povrsina_zahvacena_ha, broj_angazovanih_vatrogasaca)
            VALUES (%s, %s, %s, %s, %s, %s, %s)
            RETURNING id
        """, (datum_pocetka, datum_gasenja, parcela_id, uzrok,
              intenzitet, povrsina, vatrogasci))
        conn.commit()
        novi_id = cur.fetchone()[0]
        print(f"Novi požar dodat sa ID: {novi_id}")
        return novi_id
    except Exception as e:
        conn.rollback()
        print(f"Greška pri dodavanju: {e}")
        return None
 
novi_id = dodaj_pozar(
    conn, cur,
    datum_pocetka="2024-09-01",
    datum_gasenja="2024-09-02",
    parcela_id=1,
    uzrok="ljudski faktor (kampovanje)",
    intenzitet="srednji",
    povrsina=15.20,
    vatrogasci=10
)
 
print("\nPožari nakon dodavanja:")
display(citaj_sve_pozare())

Novi požar dodat sa ID: 7

Požari nakon dodavanja:


,id,datum_pocetka,datum_gasenja,parcela,uzrok,intenzitet,povrsina_zahvacena_ha,broj_angazovanih_vatrogasaca
0,7,2024-09-01,2024-09-02,Parcela Vlasina,ljudski faktor (kampovanje),srednji,15.2,10
1,6,2024-08-15,2024-08-15,Parcela Crni vrh,udar groma,nizak,5.5,8
2,5,2024-08-10,2024-08-12,Parcela Stara planina sever,nepoznat,visok,55.0,22
3,4,2024-08-03,2024-08-06,Parcela Strelac,nepoznat,visok,60.0,25
4,3,2024-08-02,2024-08-05,Parcela Petrlaš,ljudski faktor (spaljivanje korova),srednji,22.7,12
5,2,2024-07-22,2024-07-22,Parcela Rsovci,udar groma,nizak,3.1,6
6,1,2024-07-10,2024-07-12,Parcela Vlasina,ljudski faktor (loza),visok,45.3,18


In [16]:
# Crud operacija update

def azuriraj_pozar(conn, cur, pozar_id, nova_povrsina, novi_intenzitet):
    if nova_povrsina <= 0:
        print("Površina mora biti pozitivna!")
        return False
    try:
        cur.execute("""
            UPDATE pozari
            SET povrsina_zahvacena_ha = %s, intenzitet = %s
            WHERE id = %s
        """, (nova_povrsina, novi_intenzitet, pozar_id))
        conn.commit()
 
        if cur.rowcount == 0:
            print(f"Požar sa ID {pozar_id} nije pronađen — ništa nije ažurirano.")
            return False
 
        print(f"Požar ID {pozar_id} ažuriran (površina={nova_povrsina}, intenzitet={novi_intenzitet})")
        return True
    except Exception as e:
        conn.rollback()
        print(f"Greška pri ažuriranju: {e}")
        return False
 
if novi_id:
    azuriraj_pozar(conn, cur, novi_id, 18.50, "visok")
 
print("\nPožari nakon ažuriranja:")
display(citaj_sve_pozare().head(3))

Požar ID 7 ažuriran (površina=18.5, intenzitet=visok)

Požari nakon ažuriranja:


,id,datum_pocetka,datum_gasenja,parcela,uzrok,intenzitet,povrsina_zahvacena_ha,broj_angazovanih_vatrogasaca
0,7,2024-09-01,2024-09-02,Parcela Vlasina,ljudski faktor (kampovanje),visok,18.5,10
1,6,2024-08-15,2024-08-15,Parcela Crni vrh,udar groma,nizak,5.5,8
2,5,2024-08-10,2024-08-12,Parcela Stara planina sever,nepoznat,visok,55.0,22


In [17]:
# CRUD OPERACIJE - DELETE

def obrisi_pozar(conn, cur, pozar_id):
    try:
        cur.execute("DELETE FROM pozari WHERE id = %s", (pozar_id,))
        conn.commit()
 
        if cur.rowcount == 0:
            print(f"Požar sa ID {pozar_id} nije pronađen — ništa nije obrisano.")
            return False
 
        print(f"Požar ID {pozar_id} obrisan (i sva povezana opožarena područja, CASCADE)")
        return True
    except Exception as e:
        conn.rollback()
        print(f"Greška pri brisanju: {e}")
        return False
 
if novi_id:
    obrisi_pozar(conn, cur, novi_id)
 
print("\nPožari nakon brisanja:")
display(citaj_sve_pozare())

Požar ID 7 obrisan (i sva povezana opožarena područja, CASCADE)

Požari nakon brisanja:


,id,datum_pocetka,datum_gasenja,parcela,uzrok,intenzitet,povrsina_zahvacena_ha,broj_angazovanih_vatrogasaca
0,6,2024-08-15,2024-08-15,Parcela Crni vrh,udar groma,nizak,5.5,8
1,5,2024-08-10,2024-08-12,Parcela Stara planina sever,nepoznat,visok,55.0,22
2,4,2024-08-03,2024-08-06,Parcela Strelac,nepoznat,visok,60.0,25
3,3,2024-08-02,2024-08-05,Parcela Petrlaš,ljudski faktor (spaljivanje korova),srednji,22.7,12
4,2,2024-07-22,2024-07-22,Parcela Rsovci,udar groma,nizak,3.1,6
5,1,2024-07-10,2024-07-12,Parcela Vlasina,ljudski faktor (loza),visok,45.3,18


In [18]:
# JOIN UPITI - 5 PRIMERA
print("=" * 60)
print("JOIN 1 — Požari sa podacima o šumskim parcelama")
print("=" * 60)
query_j1 = """
    SELECT p.id, p.datum_pocetka, p.intenzitet,
           s.naziv as parcela, s.tip_sume, s.povrsina_ha
    FROM pozari p
    JOIN sumske_parcele s ON p.parcela_id = s.id
    ORDER BY p.datum_pocetka DESC
"""
print(pd.read_sql(query_j1, conn))
 
print("\n" + "=" * 60)
print("JOIN 2 — Požari sa brojem vatrogasaca i opštinom")
print("=" * 60)
query_j2 = """
    SELECT p.id, p.datum_pocetka, p.broj_angazovanih_vatrogasaca,
           o.naziv as opstina, o.region
    FROM pozari p
    JOIN sumske_parcele s ON p.parcela_id = s.id
    JOIN opstine o ON s.opstina_id = o.id
    ORDER BY p.broj_angazovanih_vatrogasaca DESC
"""
print(pd.read_sql(query_j2, conn))
 
print("\n" + "=" * 60)
print("JOIN 3 — Meteo merenja na dan požara (vezano po opštini)")
print("=" * 60)
# ISPRAVKA: meteo se vezuje za ISTU opštinu u kojoj je požar
query_j3 = """
    SELECT p.id as pozar_id, p.datum_pocetka,
           p.intenzitet, o.naziv as opstina,
           AVG(m.temperatura)      as avg_temperatura,
           AVG(m.vlaznost_vazduha) as avg_vlaznost,
           AVG(m.brzina_vetra)     as avg_vetar
    FROM pozari p
    JOIN sumske_parcele s  ON p.parcela_id  = s.id
    JOIN opstine o         ON s.opstina_id  = o.id
    JOIN meteo_stanice ms  ON ms.opstina_id = o.id
    JOIN meteo_merenja m   ON m.stanica_id  = ms.id
                          AND m.datum       = p.datum_pocetka
    GROUP BY p.id, p.datum_pocetka, p.intenzitet, o.naziv
    ORDER BY p.datum_pocetka
"""
print(pd.read_sql(query_j3, conn))
 
print("\n" + "=" * 60)
print("JOIN 4 — Ukupna površina požara po opštinama")
print("=" * 60)
query_j4 = """
    SELECT o.naziv as opstina,
           COUNT(p.id)                          as broj_pozara,
           SUM(p.povrsina_zahvacena_ha)         as ukupna_povrsina_ha,
           ROUND(AVG(p.broj_angazovanih_vatrogasaca), 1) as prosecno_vatrogasaca
    FROM opstine o
    LEFT JOIN sumske_parcele s ON o.id = s.opstina_id
    LEFT JOIN pozari p         ON s.id = p.parcela_id
    GROUP BY o.naziv
    ORDER BY ukupna_povrsina_ha DESC NULLS LAST
"""
print(pd.read_sql(query_j4, conn))
 
print("\n" + "=" * 60)
print("JOIN 5 — Opožarena područja sa detaljima požara")
print("=" * 60)
query_j5 = """
    SELECT op.datum_detekcije, op.povrsina_ha, op.indeks_dnbr,
           p.datum_pocetka, p.intenzitet, p.uzrok,
           s.naziv as parcela
    FROM opozarena_podrucja op
    JOIN pozari p          ON op.pozar_id   = p.id
    JOIN sumske_parcele s  ON p.parcela_id  = s.id
    ORDER BY op.povrsina_ha DESC
"""
print(pd.read_sql(query_j5, conn))

JOIN 1 — Požari sa podacima o šumskim parcelama
   id datum_pocetka intenzitet                      parcela       tip_sume  \
0   6    2024-08-15      nizak             Parcela Crni vrh    borova šuma   
1   5    2024-08-10      visok  Parcela Stara planina sever  mesovita šuma   
2   4    2024-08-03      visok              Parcela Strelac    bukova šuma   
3   3    2024-08-02    srednji              Parcela Petrlaš    borova šuma   
4   2    2024-07-22      nizak               Parcela Rsovci  mesovita šuma   
5   1    2024-07-10      visok              Parcela Vlasina    bukova šuma   

   povrsina_ha  
0        198.4  
1        410.0  
2        275.8  
3        150.0  
4        210.2  
5        340.5  

JOIN 2 — Požari sa brojem vatrogasaca i opštinom
   id datum_pocetka  broj_angazovanih_vatrogasaca       opstina  \
0   4    2024-08-03                            25  Dimitrovgrad   
1   5    2024-08-10                            22     Knjaževac   
2   1    2024-07-10                

In [19]:
print("=" * 60)
print("WHERE 1 — Požari visokog intenziteta, površina > 20ha")
print("=" * 60)
query_w1 = """
    SELECT p.id, p.datum_pocetka, p.intenzitet,
           p.povrsina_zahvacena_ha,
           s.naziv as parcela, s.tip_sume, o.naziv as opstina
    FROM pozari p
    JOIN sumske_parcele s ON p.parcela_id  = s.id
    JOIN opstine o        ON s.opstina_id  = o.id
    WHERE p.intenzitet = 'visok'
      AND p.povrsina_zahvacena_ha > 20
    ORDER BY p.povrsina_zahvacena_ha DESC
"""
print(pd.read_sql(query_w1, conn))
 
print("\n" + "=" * 60)
print("WHERE 2 — Požari koji su trajali više od 2 dana")
print("=" * 60)
query_w2 = """
    SELECT p.id, p.datum_pocetka, p.datum_gasenja,
           (p.datum_gasenja - p.datum_pocetka) as trajanje_dana,
           p.povrsina_zahvacena_ha,
           s.naziv as parcela
    FROM pozari p
    JOIN sumske_parcele s ON p.parcela_id = s.id
    WHERE p.datum_gasenja IS NOT NULL
      AND (p.datum_gasenja - p.datum_pocetka) > 2
    ORDER BY trajanje_dana DESC
"""
print(pd.read_sql(query_w2, conn))
 
print("\n" + "=" * 60)
print("WHERE 3 — Meteo na dan požara (temp > 33°C, ista opština)")
print("=" * 60)
query_w3 = """
    SELECT p.id as pozar_id, p.datum_pocetka,
           p.intenzitet, p.povrsina_zahvacena_ha,
           m.temperatura, m.vlaznost_vazduha, m.brzina_vetra,
           ms.naziv as meteo_stanica, o.naziv as opstina
    FROM pozari p
    JOIN sumske_parcele s ON p.parcela_id  = s.id
    JOIN opstine o        ON s.opstina_id  = o.id
    JOIN meteo_stanice ms ON ms.opstina_id = o.id
    JOIN meteo_merenja m  ON m.stanica_id  = ms.id
                         AND m.datum       = p.datum_pocetka
    WHERE m.temperatura > 33
    ORDER BY m.temperatura DESC
"""
print(pd.read_sql(query_w3, conn))
 
print("\n" + "=" * 60)
print("WHERE 4 — Požari u opštini Pirot, vatrogasci > 10")
print("=" * 60)
query_w4 = """
    SELECT p.id, p.datum_pocetka, p.intenzitet,
           p.broj_angazovanih_vatrogasaca,
           p.povrsina_zahvacena_ha, o.naziv as opstina
    FROM pozari p
    JOIN sumske_parcele s ON p.parcela_id  = s.id
    JOIN opstine o        ON s.opstina_id  = o.id
    WHERE o.naziv = 'Pirot'
      AND p.broj_angazovanih_vatrogasaca > 10
    ORDER BY p.broj_angazovanih_vatrogasaca DESC
"""
print(pd.read_sql(query_w4, conn))
 
print("\n" + "=" * 60)
print("WHERE 5 — Opožarena područja, dNBR > 0.40")
print("=" * 60)
query_w5 = """
    SELECT op.id, op.datum_detekcije,
           op.povrsina_ha, op.indeks_dnbr,
           p.datum_pocetka, p.intenzitet, s.naziv as parcela
    FROM opozarena_podrucja op
    JOIN pozari p         ON op.pozar_id  = p.id
    JOIN sumske_parcele s ON p.parcela_id = s.id
    WHERE op.indeks_dnbr > 0.40
    ORDER BY op.indeks_dnbr DESC
"""
print(pd.read_sql(query_w5, conn))
 
print("\n" + "=" * 60)
print("WHERE 6 — Opštine sa ukupnom površinom požara > 50ha (HAVING)")
print("=" * 60)
query_w6 = """
    SELECT o.naziv as opstina,
           COUNT(p.id)                  as broj_pozara,
           SUM(p.povrsina_zahvacena_ha) as ukupna_povrsina_ha,
           ROUND(AVG(p.broj_angazovanih_vatrogasaca), 1) as prosecno_vatrogasaca
    FROM opstine o
    JOIN sumske_parcele s ON o.id = s.opstina_id
    LEFT JOIN pozari p    ON s.id = p.parcela_id
    GROUP BY o.naziv
    HAVING SUM(p.povrsina_zahvacena_ha) > 50
    ORDER BY ukupna_povrsina_ha DESC
"""
print(pd.read_sql(query_w6, conn))
 
print("\n" + "=" * 60)
print("WHERE 7 — Šumske parcele sa bar jednim požarom (HAVING)")
print("=" * 60)
query_w7 = """
    SELECT s.naziv as parcela, s.tip_sume,
           COUNT(p.id)                  as broj_pozara,
           SUM(p.povrsina_zahvacena_ha) as ukupna_povrsina
    FROM sumske_parcele s
    LEFT JOIN pozari p ON s.id = p.parcela_id
    GROUP BY s.naziv, s.tip_sume
    HAVING COUNT(p.id) >= 1
    ORDER BY broj_pozara DESC
"""
print(pd.read_sql(query_w7, conn))

WHERE 1 — Požari visokog intenziteta, površina > 20ha
   id datum_pocetka intenzitet  povrsina_zahvacena_ha  \
0   4    2024-08-03      visok                   60.0   
1   5    2024-08-10      visok                   55.0   
2   1    2024-07-10      visok                   45.3   

                       parcela       tip_sume       opstina  
0              Parcela Strelac    bukova šuma  Dimitrovgrad  
1  Parcela Stara planina sever  mesovita šuma     Knjaževac  
2              Parcela Vlasina    bukova šuma         Pirot  

WHERE 2 — Požari koji su trajali više od 2 dana
   id datum_pocetka datum_gasenja  trajanje_dana  povrsina_zahvacena_ha  \
0   3    2024-08-02    2024-08-05              3                   22.7   
1   4    2024-08-03    2024-08-06              3                   60.0   

           parcela  
0  Parcela Petrlaš  
1  Parcela Strelac  

WHERE 3 — Meteo na dan požara (temp > 33°C, ista opština)
   pozar_id datum_pocetka intenzitet  povrsina_zahvacena_ha  temperatura

In [20]:
# JOIN + GROUP BY + HAVING (filtriranje grupa)

print("=" * 60)
print("UPIT 6: Opštine sa ukupnom površinom požara preko 50ha")
print("=" * 60)

query6 = """
    SELECT o.naziv as opstina, 
           COUNT(p.id) as broj_pozara,
           SUM(p.povrsina_zahvacena_ha) as ukupna_povrsina_ha,
           AVG(p.broj_angazovanih_vatrogasaca) as prosecno_vatrogasaca
    FROM opstine o
    JOIN sumske_parcele s ON o.id = s.opstina_id
    LEFT JOIN pozari p ON s.id = p.parcela_id
    GROUP BY o.naziv
    HAVING SUM(p.povrsina_zahvacena_ha) > 50
    ORDER BY ukupna_povrsina_ha DESC
"""
print(pd.read_sql(query6, conn))


print("\n" + "=" * 60)
print("UPIT 7: Šumske parcele sa više od 1 požara")
print("=" * 60)

query7 = """
    SELECT s.naziv as parcela, s.tip_sume,
           COUNT(p.id) as broj_pozara,
           SUM(p.povrsina_zahvacena_ha) as ukupna_povrsina
    FROM sumske_parcele s
    LEFT JOIN pozari p ON s.id = p.parcela_id
    GROUP BY s.naziv, s.tip_sume
    HAVING COUNT(p.id) >= 1
    ORDER BY broj_pozara DESC
"""
print(pd.read_sql(query7, conn))

UPIT 6: Opštine sa ukupnom površinom požara preko 50ha
        opstina  broj_pozara  ukupna_povrsina_ha  prosecno_vatrogasaca
0  Dimitrovgrad            1                60.0                  25.0
1     Knjaževac            1                55.0                  22.0

UPIT 7: Šumske parcele sa više od 1 požara
                       parcela       tip_sume  broj_pozara  ukupna_povrsina
0  Parcela Stara planina sever  mesovita šuma            1             55.0
1               Parcela Rsovci  mesovita šuma            1              3.1
2              Parcela Strelac    bukova šuma            1             60.0
3             Parcela Crni vrh    borova šuma            1              5.5
4              Parcela Petrlaš    borova šuma            1             22.7
5              Parcela Vlasina    bukova šuma            1             45.3


In [21]:
# ČUVANJE PODATAKA U DATAFRAME

# Učitavamo sve tabele u DataFrame-ove
df_opstine    = pd.read_sql("SELECT * FROM opstine",             conn)
df_parcele    = pd.read_sql("SELECT * FROM sumske_parcele",      conn)
df_vatrogasne = pd.read_sql("SELECT * FROM vatrogasne_jedinice", conn)
df_stanice    = pd.read_sql("SELECT * FROM meteo_stanice",       conn)
df_merenja    = pd.read_sql("SELECT * FROM meteo_merenja",       conn)
df_pozari     = pd.read_sql("SELECT * FROM pozari",              conn)
df_opozarena  = pd.read_sql("SELECT * FROM opozarena_podrucja",  conn)
 
print("Svi podaci učitani u DataFrame-ove:")
for naziv, df in [("opstine", df_opstine), ("sumske_parcele", df_parcele),
                  ("vatrogasne_jedinice", df_vatrogasne), ("meteo_stanice", df_stanice),
                  ("meteo_merenja", df_merenja), ("pozari", df_pozari),
                  ("opozarena_podrucja", df_opozarena)]:
    print(f"   {naziv}: {len(df)} redova")
 

Svi podaci učitani u DataFrame-ove:
   opstine: 5 redova
   sumske_parcele: 6 redova
   vatrogasne_jedinice: 5 redova
   meteo_stanice: 6 redova
   meteo_merenja: 918 redova
   pozari: 6 redova
   opozarena_podrucja: 6 redova


In [22]:
cur.close()
conn.close()
print("Konekcija zatvorena")

Konekcija zatvorena
